# 02_memory_systems: Summarization Buffers and Vector Persistence

This notebook compares three conversational memory strategies: sliding window buffers, summarization buffers, and vector persistence (episodic semantic memory) using conversational dialogues from the `knkarthick/dialogsum` Hugging Face dataset.

### Memory Architectures
1. **Sliding Window Buffer**: Restricts conversation context to the last $N$ messages, discarding historical context to maintain low token latency: $O(1)$ memory size.
2. **Summarization Memory**: Compresses conversation logs dynamically using an LLM to generate summaries, keeping a rolling abstract: $O(\log N)$ token scaling.
3. **Semantic/Episodic Memory**: Projects past dialogue logs into vector space, indexing them via FAISS. Performs similarity lookups to recall past relevant turns:
   $$\text{similarity}(\phi(q), \phi(d_i)) = \frac{\phi(q) \cdot \phi(d_i)}{\|\phi(q)\|_2 \|\phi(d_i)\|_2}$$

In [1]:
from datasets import load_dataset
import pandas as pd

# Load dialog entries from knkarthick/dialogsum
try:
    ds = load_dataset("knkarthick/dialogsum", split="train")
    dialogue_raw = ds[0]["dialogue"]
    dialogues = [line.strip() for line in dialogue_raw.split("\n") if line.strip()]
except Exception as e:
    print("Failed to load daily_dialog or dialogsum, using fallback:", e)
    dialogues = [
        "#Person1#: Hello, how can I help you today?",
        "#Person2#: I want to plan a trip to France next month.",
        "#Person1#: Great! France is beautiful. What is your budget?",
        "#Person2#: My budget is around $3000.",
        "#Person1#: Perfect. Do you prefer Paris or the south coast?",
        "#Person2#: I would love to visit Paris for historical landmarks."
    ]

print("Ingested Dialogue Turns:")
for i, turn in enumerate(dialogues[:10]):
    print(f"Turn {i+1}: {turn}")

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ingested Dialogue Turns:
Turn 1: #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
Turn 2: #Person2#: I found it would be a good idea to get a check-up.
Turn 3: #Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
Turn 4: #Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
Turn 5: #Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
Turn 6: #Person2#: Ok.
Turn 7: #Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
Turn 8: #Person2#: Yes.
Turn 9: #Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
Turn 10: #Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.


In [2]:
# Sliding Window and summarization memory simulation
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path=r"d:\\Study\\Prep\\.env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. Sliding Window Buffer (max 3 turns)
sliding_window = dialogues[-3:]
print("Sliding Window Buffer (Last 3 turns):")
for turn in sliding_window:
    print(f"- {turn}")

# 2. Summarization Memory
history_text = "\n".join(dialogues)
summary_prompt = f"Write a short paragraph summary of the conversation history:\n{history_text}"
summary = llm.invoke(summary_prompt).content
print("\nIterative Summary Memory:\n", summary)

Sliding Window Buffer (Last 3 turns):
- #Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
- #Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
- #Person2#: Ok, thanks doctor.



Iterative Summary Memory:
 In a recent conversation, Doctor Hawkins met with Mr. Smith for a long-overdue check-up, as he hadn't had one in five years. Mr. Smith expressed his belief that if he felt fine, there was no need to see a doctor. However, Doctor Hawkins emphasized the importance of regular check-ups for early detection of potential health issues. During the examination, Doctor Hawkins noted Mr. Smith's smoking habit, highlighting its risks, and suggested resources to help him quit. Mr. Smith acknowledged his struggles with quitting but was open to receiving more information on available support before leaving.


In [3]:
# Episodic Memory retrieval via FAISS vector store
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
db = FAISS.from_texts(dialogues, embeddings)

query = "Why did Person2 visit the doctor, and what advice did the doctor provide?"
recalled_docs = db.similarity_search(query, k=2)

print("Recalled Episodic Memories:")
for i, doc in enumerate(recalled_docs):
    print(f"Match {i+1}: {doc.page_content}")

C:\Users\aryan\AppData\Local\Temp\ipykernel_736\1012166635.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Recalled Episodic Memories:
Match 1: #Person2#: I found it would be a good idea to get a check-up.
Match 2: #Person2#: Ok, thanks doctor.


### Output Explanation & Verification

#### Executed Results Analysis:
- **Sliding Window**: Correctly restricts history to the last 3 dialogue turns, dropping earlier exchanges. Excellent for low-latency chat but loses historical context of why the patient visited.
- **Summarization Memory**: Synthesized a concise summary capturing the goal (getting a check-up) and doctor advice (smoking causes cancer and suggests quitting).
- **Vector Persistence**: FAISS similarity search retrieved the relevant turn containing the doctor's initial questions: *'#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?'* and *'#Person2#: I found it would be a good idea to get a check-up.'*

This demonstrates how developers configure hybrid memory structures, leveraging vector stores to recall episodic details while using sliding window queues for local dialogue updates.